# Example 11.1 / 1: Write a Quadratic Objective

Companion notebook for **11.1 Quadratic Models** in the AMPL lab Documentation, section "Write a Quadratic Objective". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write a Quadratic Objective

A quadratic objective can be written

$$\min_x\ \tfrac12 x^\top Qx+c^\top x,$$

possibly with linear or quadratic constraints. For a minimization QP over a convex feasible set, a symmetric positive-semidefinite $Q$ makes the objective convex. An indefinite $Q$ produces a nonconvex problem; a negative-semidefinite $Q$ is appropriate for a concave maximization objective.

Each definiteness class has a two-variable instance whose sign can be checked by hand, using
$\tfrac12x^TQx$:

| Class of $Q$ | Example $Q$ | Eigenvalues | $\tfrac12x^TQx$ | Concrete evidence |
|---|---|---|---|---|
| positive definite | `[[2,0],[0,2]]` | 2, 2 | $x_1^2+x_2^2$ | positive at every $x\ne0$, for example $2$ at $(1,-1)$ |
| positive semidefinite | `[[1,1],[1,1]]` | 0, 2 | $\tfrac12(x_1+x_2)^2$ | $0$ at $(1,-1)$: a flat direction |
| indefinite | `[[2,0],[0,-2]]` | 2, -2 | $x_1^2-x_2^2$ | $1$ at $(1,0)$ and $-1$ at $(0,1)$ |
| negative semidefinite | `[[-2,0],[0,0]]` | -2, 0 | $-x_1^2$ | never positive; concave, so suited to `maximize` |

AMPL accepts algebraic products and powers, so the formula is written as it reads. With the
positive definite `Q` above and $c=(-2,-4)$, the objective is $x_1^2+x_2^2-2x_1-4x_2$, whose
minimizer is $(1,2)$ with value $-5$. HiGHS solves convex QPs directly:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer > 0;   # number of variables
param Q {1..n, 1..n};  # quadratic coefficients; only the symmetric part counts
param c {1..n};        # linear coefficients

var x {1..n};  # free decision vector

minimize QuadraticObjective:  # (1/2) x'Qx + c'x
    0.5 * sum {i in 1..n, j in 1..n} Q[i,j] * x[i] * x[j]
  + sum {i in 1..n} c[i] * x[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 2
ampl.param["Q"] = {(1, 1): 2, (1, 2): 0, (2, 1): 0, (2, 2): 2}
ampl.param["c"] = {1: -2, 2: -4}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print({i: round(v, 6) for i, v in ampl.var["x"].to_dict().items()})
print(round(ampl.obj["QuadraticObjective"].value(), 6))

solved
{1: 1.0, 2: 2.0}
-5.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param n := 2;
param Q: 1 2 :=
  1    2 0
  2    0 2;
param c := 1 -2  2 -4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display x, QuadraticObjective;

x [*] :=
1  1
2  2
;

QuadraticObjective = -5



### Another way in amplpy

Only the symmetric part of a coefficient matrix affects $x^TQx$, because
$x^TQx=x^T((Q+Q^T)/2)x$. Cross terms also need care: if the model contains
$a_{ij}x_ix_j$ once, the corresponding two symmetric off-diagonal entries are each
$a_{ij}/2$ in the conventional matrix form. This distinction prevents an accidental factor
of two when a Hessian or eigenvalue test is assembled outside AMPL.

The nonsymmetric matrix `[[2,3],[-3,2]]` has symmetric part `[[2,0],[0,2]]`, so it defines the
same objective. Solving the same `model.mod` with each matrix returns the same point $(1,2)$ and
value $-5$:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["n"] = 2
ampl.param["Q"] = {(1, 1): 2, (1, 2): 0, (2, 1): 0, (2, 2): 2}
ampl.param["c"] = {1: -2, 2: -4}

In [11]:
# run.py
for name, q in (("nonsymmetric", {(1, 1): 2, (1, 2): 3, (2, 1): -3, (2, 2): 2}),
                ("symmetric part", {(1, 1): 2, (1, 2): 0, (2, 1): 0, (2, 2): 2})):
    ampl.param["Q"] = q
    ampl.solve(solver="highs", verbose=False)
    x = ampl.var["x"].to_dict()
    print(name, round(x[1], 6), round(x[2], 6),
          round(ampl.obj["QuadraticObjective"].value(), 6))

nonsymmetric 1.0 2.0 -5.0
symmetric part 1.0 2.0 -5.0
